In [1]:
# ============================================================
# Cellule 1 : Configuration de l'environnement et Repo
# ============================================================
import os
import shutil
import torch

# 1. Vérification du GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"--- 🚀 Exécution sur : {device.upper()} ---")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    print(f"GPU détecté : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ ATTENTION : Le GPU n'est pas activé.")

# 2. Gestion du dossier projet (Clonage du repo si nécessaire)
repo_url = "https://github.com/Lduvignacq/Deep-learning-project.git"
repo_dir = "Deep-learning-project"

if not os.path.exists(repo_dir):
    print(f"Clonage du repo : {repo_url}")
    !git clone {repo_url}
else:
    print(f"Le dossier {repo_dir} existe déjà.")

os.chdir(repo_dir)
print(f"Répertoire de travail actuel : {os.getcwd()}")

In [2]:
# ============================================================
# Cellule 2 : Installation des dépendances
# ============================================================
!pip install --upgrade pip -q
!pip install albumentations opencv-python pandas matplotlib tqdm -q
# Note : pretrainedmodels ou timm pourraient être requis par le repo
!pip install pretrainedmodels -q

In [3]:
# ============================================================
# Cellule 3 : Monter Google Drive et préparer le dataset APPA-REAL
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os

# 👉 Chemin vers appa-real-release.zip dans votre Drive
zip_path = '/content/drive/MyDrive/appa-real-release.zip'

if os.path.exists(zip_path):
    print("📦 Copie et décompression du dataset depuis Google Drive...")
    !cp /content/drive/MyDrive/appa-real-release.zip .
    # -o : overwrite, -q : silencieux
    !unzip -o -q appa-real-release.zip
    # Supprime les fichiers macOS cachés
    !rm -rf __MACOSX
    print("✅ Dataset prêt !")
    !ls -la appa-real-release/
else:
    print(f"❌ Fichier non trouvé: {zip_path}")
    print("📁 Veuillez uploader appa-real-release.zip dans 'Mon Drive' d'abord.")

In [ ]:
# ============================================================
# Cellule 4 : Vérification du contenu du dataset
# ============================================================
import pandas as pd
from pathlib import Path

DATA_ROOT = Path("appa-real-release")

print("📂 Contenu de appa-real-release :")
!ls -la appa-real-release | head -n 20

print("\n📊 Répartition du dataset :")
for split in ["train", "valid", "test"]:
    csv_path = DATA_ROOT / f"gt_avg_{split}.csv"
    img_dir  = DATA_ROOT / split

    if csv_path.exists():
        df = pd.read_csv(csv_path)
        n_csv = len(df)
    else:
        print(f"  ❌ CSV manquant : {csv_path}")
        continue

    if img_dir.exists():
        n_imgs = len(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))
    else:
        n_imgs = "dossier manquant"

    total = n_csv + (0 if not isinstance(n_imgs, int) else 0)
    print(f"  {split:>5} — CSV : {n_csv:>5} lignes | Images : {n_imgs}")

total_csv = sum(
    len(pd.read_csv(DATA_ROOT / f"gt_avg_{s}.csv"))
    for s in ["train", "valid", "test"]
    if (DATA_ROOT / f"gt_avg_{s}.csv").exists()
)
print(f"\n  Total : {total_csv} images")

In [ ]:
# ============================================================
# Cellule 5 : Dataset APPA-REAL pour RESIDUAL DEX
# ============================================================
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

DATA_ROOT = Path("appa-real-release")
print("DATA_ROOT:", DATA_ROOT.resolve())

class ResidualAgeDataset(Dataset):
    """
    Dataset APPA-REAL pour la méthode Residual DEX.
    - Apparent Age : pour la tête de classification (DEX).
    - Real Age : pour la tête de régression résiduelle.
    """
    def __init__(self, csv_path, img_dir, transform=None, num_classes=101):
        self.csv_path = Path(csv_path)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.num_classes = num_classes

        if not self.csv_path.exists():
            raise FileNotFoundError(f"CSV non trouvé: {self.csv_path}")
        if not self.img_dir.exists():
            raise FileNotFoundError(f"Dossier images non trouvé: {self.img_dir}")

        self.df = pd.read_csv(self.csv_path)

        if "file_name" not in self.df.columns:
            raise ValueError(f"Colonne 'file_name' absente dans {self.csv_path}")

        # Certaines versions ont 'apparent_age', d'autres 'apparent_age_avg'
        if "apparent_age" in self.df.columns:
            self.app_age_col = "apparent_age"
        elif "apparent_age_avg" in self.df.columns:
            self.app_age_col = "apparent_age_avg"
        else:
            raise ValueError(
                f"Colonne 'apparent_age' ou 'apparent_age_avg' absente dans {self.csv_path}"
            )

        if "real_age" not in self.df.columns:
            raise ValueError(f"Colonne 'real_age' absente dans {self.csv_path}")

        self.real_age_col = "real_age"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["file_name"]
        app_age = float(row[self.app_age_col])
        real_age = float(row[self.real_age_col])

        img_path = self.img_dir / img_name
        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f"Image non trouvée: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            img = self.transform(image=img)["image"]

        # Classe d'âge pour DEX (apparent)
        age_class = int(round(app_age))
        age_class = max(0, min(self.num_classes - 1, age_class))

        return img, age_class, app_age, real_age


# --- Augmentations ---
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

valid_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# --- Datasets ---
NUM_CLASSES = 101
BATCH_SIZE = 32

train_dataset = ResidualAgeDataset(DATA_ROOT / "gt_avg_train.csv", DATA_ROOT / "train", transform=train_transform, num_classes=NUM_CLASSES)
valid_dataset = ResidualAgeDataset(DATA_ROOT / "gt_avg_valid.csv", DATA_ROOT / "valid", transform=valid_transform, num_classes=NUM_CLASSES)

# --- Dataloaders ---
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} images")
print(f"Valid: {len(valid_dataset)} images")

In [ ]:
# ============================================================
# Cellule 6 : Modèle RESIDUAL DEX
# ============================================================
import torch
import torch.nn as nn
import sys, os
from model import get_model

class ResidualDEXModel(nn.Module):
    """
    Modèle Residual DEX :
    1.Une branche DEX pour prédire l'âge apparent (classification 101 classes).
    2.Une branche résiduelle pour prédire l'écart (Real_Age - Apparent_Age).
    """
    def __init__(self, num_classes=101, cnn_model_name="se_resnext50_32x4d", pretrained="imagenet"):
        super().__init__()
        
        # Backbone issu du repo
        self.backbone = get_model(model_name=cnn_model_name, num_classes=num_classes, pretrained=pretrained)
        dim_feats = self.backbone.last_linear.in_features
        
        # On remplace last_linear par un extracteur de features
        self.backbone.last_linear = nn.Identity() 
        
        # Branche 1 : DEX (Apparent Age)
        self.dex_head = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(dim_feats, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes)
        )

        # Branche 2 : Residual (Correction vers Real Age)
        self.residual_head = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(dim_feats, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 1) # Sortie scalaire
        )
        
        self.num_classes = num_classes

    def forward(self, x):
        feats = self.backbone(x)
        logits_apparent = self.dex_head(feats)
        probs_apparent = torch.softmax(logits_apparent, dim=1)
        residual = self.residual_head(feats).squeeze(1) 
        return logits_apparent, probs_apparent, residual

def calculate_expected_age(probs, device):
    age_range = torch.arange(0, probs.size(1), dtype=torch.float32, device=device)
    return torch.sum(probs * age_range.unsqueeze(0), dim=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResidualDEXModel().to(device)

# Geler définitivement le backbone
for param in model.backbone.parameters():
    param.requires_grad = False
print(f"Backbone gelé — paramètres entraînables : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Residual DEX Model chargé sur {device}")

In [ ]:
# ============================================================
# Cellule 7 : Boucles d'apprentissage Residual DEX
# ============================================================
import torch.optim as optim
from torch import amp
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, criterion_dex, criterion_res, device, scaler, alpha=1.0, beta=1.0):
    model.train()
    running_loss_dex = 0.0
    running_loss_res = 0.0

    for imgs, age_class, app_age, real_age in tqdm(loader, leave=False):
        imgs, age_class = imgs.to(device), age_class.to(device)
        app_age = app_age.float().to(device)
        real_age = real_age.float().to(device)

        optimizer.zero_grad()

        with amp.autocast('cuda' if device.type == 'cuda' else 'cpu'):
            logits_app, probs_app, residual = model(imgs)
            loss_dex = criterion_dex(logits_app, age_class)
            pred_app_age = calculate_expected_age(probs_app, device)
            pred_real_age = pred_app_age + residual
            loss_res = criterion_res(pred_real_age, real_age)
            total_loss = alpha * loss_dex + beta * loss_res

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss_dex += loss_dex.item()
        running_loss_res += loss_res.item()

    n = len(loader)
    return running_loss_dex / n, running_loss_res / n


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    mae_app = 0.0
    mae_real = 0.0

    for imgs, age_class, app_age, real_age in loader:
        imgs = imgs.to(device)
        app_age = app_age.float().to(device)
        real_age = real_age.float().to(device)

        logits_app, probs_app, residual = model(imgs)
        pred_app_age = calculate_expected_age(probs_app, device)
        pred_real_age = pred_app_age + residual

        mae_app += torch.abs(pred_app_age - app_age).sum().item()
        mae_real += torch.abs(pred_real_age - real_age).sum().item()

    n = len(loader.dataset)
    return mae_app / n, mae_real / n


def set_trainable(model, phase):
    if phase == 1:
        for param in model.dex_head.parameters():
            param.requires_grad = True
        for param in model.residual_head.parameters():
            param.requires_grad = False
    elif phase == 2:
        for param in model.dex_head.parameters():
            param.requires_grad = False
        for param in model.residual_head.parameters():
            param.requires_grad = True
    elif phase == 3:
        for param in model.dex_head.parameters():
            param.requires_grad = True
        for param in model.residual_head.parameters():
            param.requires_grad = True
    # backbone : jamais touché

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Paramètres entraînables : {trainable:,}")

print("Boucles prêtes.")

In [ ]:
# Cellule : Epochs par phase
EPOCHS_PHASE1 = 10   # DEX seul
EPOCHS_PHASE2 = 5   # Résidu seul
EPOCHS_PHASE3 = 5  # Fine-tuning global

In [ ]:
# ============================================================
# Cellule 8 : Entraînement du modèle en 3 phases
# ============================================================
import math

criterion_dex = nn.CrossEntropyLoss()
criterion_res = nn.HuberLoss()
scaler = amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

best_mae_real = math.inf

# Historique pour les courbes
history = {
    "epoch": [],
    "phase": [],
    "loss_dex": [],
    "loss_res": [],
    "mae_app": [],
    "mae_real": [],
}

epoch_global = 0  # compteur global d'époques pour l'axe x du plot

# ── PHASE 1 : DEX seul ───────────────────────────────────────
print("\n========== PHASE 1 : DEX seul ==========")
set_trainable(model, phase=1)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4
)

for epoch in range(1, EPOCHS_PHASE1 + 1):
    epoch_global += 1
    loss_dex, loss_res = train_one_epoch(
        model, train_loader, optimizer, criterion_dex, criterion_res,
        device, scaler, alpha=1.0, beta=0.0
    )
    mae_app, mae_real = validate(model, valid_loader, device)

    history["epoch"].append(epoch_global)
    history["phase"].append(1)
    history["loss_dex"].append(loss_dex)
    history["loss_res"].append(loss_res)
    history["mae_app"].append(mae_app)
    history["mae_real"].append(mae_real)

    print(f"  [P1] Epoch {epoch}/{EPOCHS_PHASE1} | Loss DEX: {loss_dex:.4f} | MAE Apparent: {mae_app:.2f} | MAE Real: {mae_real:.2f}")

torch.save(model.state_dict(), "phase1_dex.pth")
print("✅ Phase 1 terminée.")


# ── PHASE 2 : Résidu seul ────────────────────────────────────
print("\n========== PHASE 2 : Résidu seul ==========")
set_trainable(model, phase=2)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4
)

for epoch in range(1, EPOCHS_PHASE2 + 1):
    epoch_global += 1
    loss_dex, loss_res = train_one_epoch(
        model, train_loader, optimizer, criterion_dex, criterion_res,
        device, scaler, alpha=0.0, beta=1.0
    )
    mae_app, mae_real = validate(model, valid_loader, device)

    history["epoch"].append(epoch_global)
    history["phase"].append(2)
    history["loss_dex"].append(loss_dex)
    history["loss_res"].append(loss_res)
    history["mae_app"].append(mae_app)
    history["mae_real"].append(mae_real)

    print(f"  [P2] Epoch {epoch}/{EPOCHS_PHASE2} | Loss Résidu: {loss_res:.4f} | MAE Apparent: {mae_app:.2f} | MAE Real: {mae_real:.2f}")

    if mae_real < best_mae_real:
        best_mae_real = mae_real
        torch.save(model.state_dict(), "residual_dex_best.pth")
        print(f"  Nouveau meilleur modèle obtenu (MAE Real: {best_mae_real:.2f})")

print("✅ Phase 2 terminée.")


# ── PHASE 3 : Fine-tuning global ─────────────────────────────
print("\n========== PHASE 3 : Fine-tuning global ==========")
set_trainable(model, phase=3)
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

for epoch in range(1, EPOCHS_PHASE3 + 1):
    epoch_global += 1
    loss_dex, loss_res = train_one_epoch(
        model, train_loader, optimizer, criterion_dex, criterion_res,
        device, scaler, alpha=1.0, beta=1.0
    )
    mae_app, mae_real = validate(model, valid_loader, device)

    history["epoch"].append(epoch_global)
    history["phase"].append(3)
    history["loss_dex"].append(loss_dex)
    history["loss_res"].append(loss_res)
    history["mae_app"].append(mae_app)
    history["mae_real"].append(mae_real)

    print(f"  [P3] Epoch {epoch}/{EPOCHS_PHASE3} | Loss DEX: {loss_dex:.4f} | Loss Résidu: {loss_res:.4f} | MAE Apparent: {mae_app:.2f} | MAE Real: {mae_real:.2f}")

    if mae_real < best_mae_real:
        best_mae_real = mae_real
        torch.save(model.state_dict(), "residual_dex_best.pth")
        print(f"  Nouveau meilleur modèle obtenu (MAE Real: {best_mae_real:.2f})")

print(f"\nEntraînement terminé. Meilleur MAE Real: {best_mae_real:.2f}")

In [ ]:
# ============================================================
# Cellule 9 : Évaluation finale sur le set de test
# ============================================================
test_loader = DataLoader(
    ResidualAgeDataset(DATA_ROOT / "gt_avg_test.csv", DATA_ROOT / "test", transform=valid_transform),
    batch_size=BATCH_SIZE, shuffle=False
)

model.load_state_dict(torch.load("residual_dex_best.pth"))
m_app, m_real = validate(model, test_loader, device)

print(f"\n--- RÉSULTATS TEST ---")
print(f"MAE Âge Apparent : {m_app:.2f}")
print(f"MAE Âge Réel     : {m_real:.2f}")

In [ ]:
# ============================================================
# Cellule 10 : Courbes de perte + Visualisation des prédictions
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Couleurs par phase
phase_colors = {1: "steelblue", 2: "darkorange", 3: "seagreen"}
phase_labels = {1: "Phase 1 (DEX)", 2: "Phase 2 (Résidu)", 3: "Phase 3 (Fine-tuning)"}

epochs     = history["epoch"]
phases     = history["phase"]
loss_dex   = history["loss_dex"]
loss_res   = history["loss_res"]
mae_app    = history["mae_app"]
mae_real   = history["mae_real"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Graphe 1 : Loss DEX ──
ax = axes[0]
for i in range(len(epochs)):
    color = phase_colors[phases[i]]
    if i < len(epochs) - 1 and phases[i] == phases[i+1]:
        ax.plot([epochs[i], epochs[i+1]], [loss_dex[i], loss_dex[i+1]], color=color, linewidth=2)
    ax.scatter(epochs[i], loss_dex[i], color=color, s=50, zorder=5)
ax.set_title("Loss DEX (CrossEntropy)", fontsize=13)
ax.set_xlabel("epoch globale")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)

# ── Graphe 2 : Loss Résidu ──
ax = axes[1]
for i in range(len(epochs)):
    color = phase_colors[phases[i]]
    if i < len(epochs) - 1 and phases[i] == phases[i+1]:
        ax.plot([epochs[i], epochs[i+1]], [loss_res[i], loss_res[i+1]], color=color, linewidth=2)
    ax.scatter(epochs[i], loss_res[i], color=color, s=50, zorder=5)
ax.set_title("Loss Résidu (Huber)", fontsize=13)
ax.set_xlabel("epoch globale")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)

# ── Graphe 3 : MAE ──
ax = axes[2]
for i in range(len(epochs)):
    color = phase_colors[phases[i]]
    if i < len(epochs) - 1 and phases[i] == phases[i+1]:
        ax.plot([epochs[i], epochs[i+1]], [mae_app[i], mae_app[i+1]], color=color, linewidth=2, linestyle="--")
        ax.plot([epochs[i], epochs[i+1]], [mae_real[i], mae_real[i+1]], color=color, linewidth=2, linestyle="-")
    ax.scatter(epochs[i], mae_app[i], color=color, s=50, marker="^", zorder=5)
    ax.scatter(epochs[i], mae_real[i], color=color, s=50, marker="o", zorder=5)
ax.set_title("MAE (validation)", fontsize=13)
ax.set_xlabel("epoch globale")
ax.set_ylabel("MAE (années)")
ax.grid(True, alpha=0.3)

# Légendes
patches_phase = [mpatches.Patch(color=c, label=phase_labels[p]) for p, c in phase_colors.items()]
line_app  = plt.Line2D([0], [0], color="gray", linestyle="--", label="MAE Apparent")
line_real = plt.Line2D([0], [0], color="gray", linestyle="-",  label="MAE Réel")
axes[2].legend(handles=patches_phase + [line_app, line_real], fontsize=8)
axes[0].legend(handles=patches_phase, fontsize=8)
axes[1].legend(handles=patches_phase, fontsize=8)

plt.suptitle("Courbes d'entraînement Res DEX", fontsize=15)
plt.tight_layout()
plt.show()


# ── Visualisation des prédictions ────────────────────────────
model.load_state_dict(torch.load("residual_dex_best.pth", map_location=device))
model.eval()

def visualize_results(model, dataset, device, n=8):
    indices = np.random.choice(len(dataset), n, replace=False)

    plt.figure(figsize=(20, 10))
    for i, idx in enumerate(indices):
        img_tensor, _, app_age, real_age = dataset[idx]

        with torch.no_grad():
            img_input = img_tensor.unsqueeze(0).to(device)
            _, probs, res = model(img_input)
            pred_app = calculate_expected_age(probs, device).item()
            pred_real = pred_app + res.item()

        img_vis = img_tensor.permute(1, 2, 0).cpu().numpy()
        img_vis = img_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_vis = np.clip(img_vis, 0, 1)

        plt.subplot(2, 4, i + 1)
        plt.imshow(img_vis)
        plt.title(
            f"Réel: {real_age:.1f} / Prédit: {pred_real:.1f}\n"
            f"(Apparent réel: {app_age:.1f} / Apparent prédit: {pred_app:.1f})",
            fontsize=9
        )
        plt.axis("off")

    plt.suptitle("8 images test: Âge réel vs prédiction resDEX", fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_results(model, test_loader.dataset, device)